# Del conteo al espectro: unfolding de neutrones

Actividad práctica de 60 minutos para reconstruir el espectro de **un evento real**. El cálculo oficial sigue en C++/ROOT (`macros/deconv_CRNS.C`); Python permite seleccionar el evento, inspeccionar los datos, ejecutar ROOT y visualizar los resultados.

\[
\text{conteos}\rightarrow\text{tasas}\rightarrow R_{ij}\rightarrow
\text{semilla}\rightarrow\text{EM}\rightarrow\text{parada}\rightarrow
\text{refolding}\rightarrow\text{MC}\rightarrow\text{incertidumbre}
\]

| Minutos | Actividad | Pregunta central |
|---:|---|---|
| 0–5 | Conexión con el laboratorio | ¿Qué representa cada canal? |
| 5–20 | Datos, respuestas y modelo directo | ¿Qué predice la semilla? |
| 20–30 | Una iteración explícita | ¿Cómo corrige EM el espectro? |
| 30–48 | EM oficial, parada y refolding | ¿La solución reproduce los datos? |
| 48–55 | Monte Carlo reducido | ¿Cómo se propaga la fluctuación de los conteos? |
| 55–60 | Flujos integrales y cierre | ¿Qué magnitudes físicas obtenemos? |


## 1. Conexión con el laboratorio

Cada canal emplea el mismo tipo de detector central, pero una configuración moderadora distinta. Los moderadores cambian la energía de los neutrones antes de que alcancen el detector; por eso cada canal posee una función respuesta amplia y diferente. Ningún canal mide por sí solo un intervalo energético estrecho: el espectro se infiere usando simultáneamente todas las tasas.

**Pregunta inicial:** si el detector central es el mismo, ¿por qué cambian las tasas entre canales?

<details>
<summary>Respuesta esperada</summary>

Porque cada moderador transforma de manera diferente el campo de neutrones y produce una función respuesta distinta.
</details>


## 2. Preparar la interfaz y seleccionar un evento


In [ ]:
import importlib
from pathlib import Path
import subprocess

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd
from IPython.display import display

import classroom.python.unfolding_helpers as unfolding_helpers
importlib.reload(unfolding_helpers)  # Evita usar una versión anterior en memoria.

from classroom.python.unfolding_helpers import (
    REPO_DIR,
    em_iteration_history,
    expected_em_log,
    expected_em_output,
    expected_mc_output,
    expected_mc_stats,
    forward_fold,
    load_event_channels,
    load_response_matrix,
    load_seed_spectrum,
    load_shell_config,
    mc_run_summary,
    quantile_summary,
    root_keys,
    run_em,
    run_em_mc,
    scalar_tree_table,
    scientific_inputs,
    tree_branches,
    tree_entry_count,
    tree_names,
    vector_tree_array,
)

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
print("Repositorio:", REPO_DIR)


In [ ]:
# Esta es la única celda que el estudiante necesita modificar.
CONFIG_FILE = Path("configs/unfolding_configs/config_EM_MC_stop_LCO_15min_ISO.sh")
EVENT_ID = 2
SEED_NUMBER = 1              # 1 = primera semilla PARMA del día seleccionado
REUSE_EXISTING_RESULTS = True

config = load_shell_config(CONFIG_FILE)
main_keys = [
    "CAMPAIGN", "TIME_GRID", "NDET", "STEPS", "MAX_STEPS",
    "BIN_SEED", "CUT", "PHYLST", "SCF", "NEUFTY",
]
selection = pd.DataFrame({
    "variable": ["EVENT_ID", "SEED_NUMBER", *main_keys],
    "valor": [EVENT_ID, SEED_NUMBER, *[config[key] for key in main_keys]],
})
display(selection)


In [ ]:
inputs = scientific_inputs(CONFIG_FILE)
input_table = pd.DataFrame([
    {
        "entrada": name,
        "ruta dentro del repositorio": str(path.relative_to(REPO_DIR)),
        "disponible": path.is_file() if name != "respuestas" else path.is_dir(),
    }
    for name, path in inputs.items()
])
display(input_table)
assert input_table["disponible"].all(), "Falta al menos un input científico."


## 3. Conteos y tasas por canal

Para un tiempo vivo \(T_i\), el código usa:

\[
r_i=\frac{N_i}{T_i},\qquad
\sigma_{N_i}\simeq\sqrt{N_i},\qquad
\sigma_{r_i}\simeq\frac{\sqrt{N_i}}{T_i}.
\]

La tabla contiene solamente los canales activados por `Detectors_Array(...)` en el C++.


In [ ]:
event_table = load_event_channels(CONFIG_FILE, EVENT_ID)
display(event_table.style.format({
    "Tasa [s⁻¹]": "{:.5f}",
    "Incertidumbre [s⁻¹]": "{:.5f}",
    "Incertidumbre relativa [%]": "{:.1f}",
}))

channels = event_table["Canal"].to_numpy()
measured_rate = event_table["Tasa [s⁻¹]"].to_numpy()
measured_sigma = event_table["Incertidumbre [s⁻¹]"].to_numpy()

fig, axis = plt.subplots()
axis.errorbar(
    channels, measured_rate, yerr=measured_sigma,
    fmt="o", capsize=4, color="tab:blue", label="medición",
)
axis.set(xlabel="Canal activo", ylabel="Tasa [s⁻¹]",
         title=f"{config['CAMPAIGN']} · evento {EVENT_ID}")
axis.legend()
plt.show()


**Preguntas para discutir**

1. ¿Qué canal presenta la mayor tasa?
2. ¿Qué canal tiene la mayor incertidumbre relativa?
3. ¿Por qué un canal con pocos conteos aporta menos información estadística?


## 4. Funciones respuesta

\(R_{ij}\) representa la sensibilidad del canal \(i\) al bin energético \(j\). Primero compararemos un moderador delgado, una configuración con Pb y el detector desnudo; luego veremos todos los canales activos.


In [ ]:
response_matrix, energy_bins, response_metadata = load_response_matrix(CONFIG_FILE)
energy_mid = energy_bins["Emid"].to_numpy()
energy_width = energy_bins["Ewid"].to_numpy()
energy_edges = np.r_[energy_bins["Elower"].to_numpy(),
                     energy_bins["Eupper"].iloc[-1]]

print("Forma de R:", response_matrix.shape)
display(response_metadata[["Canal", "Configuración", "Archivo"]])


In [ ]:
representative_channels = ["D03", "D12", "D16"]
fig, axis = plt.subplots()
for channel in representative_channels:
    row = response_metadata.index[response_metadata["Canal"] == channel][0]
    label = response_metadata.loc[row, "Configuración"]
    axis.loglog(energy_mid, response_matrix[row], label=f"{channel}: {label}")
axis.set(xlabel="Energía [MeV]", ylabel="Respuesta",
         title="Funciones respuesta representativas")
axis.legend()
plt.show()


In [ ]:
positive = response_matrix[response_matrix > 1e-22]
color_norm = LogNorm(
    vmin=np.quantile(positive, 0.02),
    vmax=np.quantile(positive, 0.995),
)
fig, axis = plt.subplots(figsize=(12, 5))
mesh = axis.pcolormesh(
    energy_edges,
    np.arange(len(channels) + 1),
    response_matrix,
    shading="flat",
    norm=color_norm,
    cmap="viridis",
)
axis.set_xscale("log")
axis.set(
    xlabel="Energía [MeV]",
    ylabel="Canal activo",
    title="Matriz de funciones respuesta $R_{ij}$",
    yticks=np.arange(len(channels)) + 0.5,
    yticklabels=channels,
)
fig.colorbar(mesh, ax=axis, label="Respuesta")
plt.show()


**Preguntas para discutir**

- ¿Qué canales mantienen sensibilidad a energías más altas?
- ¿En qué zonas las respuestas se parecen demasiado?
- Si varias filas de \(R\) son muy similares, ¿qué parte del espectro estará peor restringida?


## 5. Modelo directo: forward folding

Antes del problema inverso, evaluamos una predicción:

\[
\widehat n_i=\sum_j R_{ij}\,\phi_j\,\Delta E_j.
\]

Conocidos \(\phi\) y \(R\), calcular tasas es directo. El unfolding intenta lo contrario: estimar \(\phi\) a partir de pocas tasas y respuestas amplias. Como hay menos canales que bins energéticos, diferentes espectros pueden producir tasas parecidas.


In [ ]:
seed_spectrum, water_fraction = load_seed_spectrum(CONFIG_FILE, SEED_NUMBER)
seed_flux = seed_spectrum["Flujo diferencial"].to_numpy()
seed_integral = seed_spectrum["Flujo por bin"].to_numpy()

fig, axis = plt.subplots()
axis.loglog(energy_mid, seed_integral, color="tab:green")
axis.set(
    xlabel="Energía [MeV]",
    ylabel=r"$\phi_j\,\Delta E_j$ [cm$^{-2}$ s$^{-1}$]",
    title=f"Semilla PARMA {SEED_NUMBER} · VWC={100*water_fraction:.1f} %",
)
plt.show()


In [ ]:
seed_prediction = forward_fold(response_matrix, seed_flux, energy_width)
seed_ratio = np.divide(
    seed_prediction, measured_rate,
    out=np.full_like(seed_prediction, np.nan),
    where=measured_rate > 0,
)
forward_table = event_table[["Canal", "Configuración", "Tasa [s⁻¹]",
                             "Incertidumbre [s⁻¹]"]].copy()
forward_table["Predicción semilla [s⁻¹]"] = seed_prediction
forward_table["Predicción / medición"] = seed_ratio
display(forward_table.style.format({
    "Tasa [s⁻¹]": "{:.5f}",
    "Incertidumbre [s⁻¹]": "{:.5f}",
    "Predicción semilla [s⁻¹]": "{:.5f}",
    "Predicción / medición": "{:.3f}",
}))

x = np.arange(len(channels))
fig, (top, bottom) = plt.subplots(
    2, 1, figsize=(11, 7), sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
)
top.errorbar(x, measured_rate, yerr=measured_sigma, fmt="o",
             capsize=3, label="medida")
top.plot(x, seed_prediction, "s-", label="semilla plegada")
top.set(ylabel="Tasa [s⁻¹]", title="Forward folding de la semilla")
top.legend()
bottom.axhline(1.0, color="black", linestyle="--")
bottom.plot(x, seed_ratio, "o", color="tab:orange")
bottom.set(
    ylabel="pred./med.",
    xlabel="Canal activo",
    xticks=x,
    xticklabels=channels,
)
plt.show()


Un cociente mayor que uno indica que la semilla sobreestima el canal; uno menor que uno indica que lo subestima. ¿Qué canales exigen la mayor corrección?


## 6. Una iteración explícita de EM

Esta única iteración en Python es **didáctica**; no sustituye al C++ oficial.

\[
\phi_j^{(s+1)}
=
\phi_j^{(s)}
\frac{\sum_iR_{ij}\left(n_i/\widehat n_i^{(s)}\right)}
     {\sum_iR_{ij}}.
\]

Pasos: (1) espectro actual, (2) plegamiento, (3) cocientes por canal, (4) retroproyección, (5) nuevo espectro y (6) nuevo plegamiento.


In [ ]:
# 1–2. Espectro actual y plegamiento
phi_before = seed_flux.copy()
predicted_before = forward_fold(response_matrix, phi_before, energy_width)

# 3. Factores de corrección por canal
channel_correction = measured_rate / predicted_before

# 4–5. Retroproyección y nuevo espectro
back_projection = (
    response_matrix.T @ channel_correction
) / response_matrix.sum(axis=0)
phi_after = phi_before * back_projection

# 6. Nuevo plegamiento y diagnósticos
predicted_after = forward_fold(response_matrix, phi_after, energy_width)
chi2_before = np.sum(((measured_rate - predicted_before) / measured_sigma) ** 2)
chi2_after = np.sum(((measured_rate - predicted_after) / measured_sigma) ** 2)
diff_after = (
    np.abs(phi_after * energy_width - phi_before * energy_width).sum()
    / (phi_before * energy_width).sum()
)

display(pd.DataFrame({
    "Canal": channels,
    "medida / predicción inicial": channel_correction,
}).style.format({"medida / predicción inicial": "{:.3f}"}))
print(f"χ² antes  = {chi2_before:.3f}")
print(f"χ² después= {chi2_after:.3f}")
print(f"diff       = {diff_after:.5f}")


In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(14, 5))
left.loglog(energy_mid, phi_before * energy_width,
            label="antes (semilla)")
left.loglog(energy_mid, phi_after * energy_width,
            label="después de 1 iteración")
left.set(xlabel="Energía [MeV]",
         ylabel=r"$\phi_j\,\Delta E_j$ [cm$^{-2}$ s$^{-1}$]",
         title="Actualización del espectro")
left.legend()

x = np.arange(len(channels))
right.errorbar(x, measured_rate, yerr=measured_sigma, fmt="o",
               capsize=3, label="medida")
right.plot(x, predicted_before, "s--", label="antes")
right.plot(x, predicted_after, "d-", label="después")
right.set(xlabel="Canal activo", ylabel="Tasa [s⁻¹]",
          title="Plegamiento antes y después",
          xticks=x, xticklabels=channels)
right.legend()
plt.show()


## 7. Ejecutar el EM nominal con C++/ROOT

Flujo real de ejecución:

```text
Notebook Python → helper Python → script Bash local
→ macros/deconv_CRNS.C → archivo ROOT
```

Se procesa un solo `EVENT_ID`, sin SLURM. Si el ROOT ya existe y `REUSE_EXISTING_RESULTS=True`, se reutiliza para que repetir el notebook sea rápido.


In [ ]:
em_file = expected_em_output(CONFIG_FILE, EVENT_ID)
em_log = expected_em_log(CONFIG_FILE, EVENT_ID)
command = (
    "bash local_scripts/run_event_em_local.sh "
    f"{CONFIG_FILE} {EVENT_ID}"
)
print("Comando       :", command)
print("Configuración :", CONFIG_FILE)
print("Evento        :", EVENT_ID)
print("Salida ROOT   :", em_file)
print("Log           :", em_log)

if REUSE_EXISTING_RESULTS and em_file.is_file() and em_log.is_file():
    print("\nSe reutiliza el resultado existente.")
else:
    try:
        run_em(CONFIG_FILE, EVENT_ID)
    except subprocess.CalledProcessError as error:
        print(f"\nROOT/Bash terminó con código {error.returncode}.")
        print("Revisa el log:", em_log)
        raise

assert em_file.is_file(), f"No se creó {em_file}"
assert em_log.is_file(), f"No se creó {em_log}"
print("\nEM nominal terminado correctamente.")


### Inspección del archivo ROOT generado


In [ ]:
display(pd.DataFrame(root_keys(em_file).items(),
                     columns=["Objeto", "Clase ROOT"]))
print("Árboles:", tree_names(em_file))
print("Entradas en em_loop_tree:", tree_entry_count(em_file))
display(pd.DataFrame({"Branch": tree_branches(em_file)}))

em_table = scalar_tree_table(em_file, max_rows=None)
nominal_spectra = vector_tree_array(em_file)
print("Forma de la matriz de espectros:", nominal_spectra.shape)
display(em_table.head())


In [ ]:
seed_row = SEED_NUMBER - 1
if seed_row >= len(em_table):
    raise IndexError("SEED_NUMBER no está disponible en el ROOT nominal.")

nominal_row = em_table.iloc[seed_row]
final_integral = nominal_spectra[seed_row]
final_differential = final_integral / energy_width
diff_branch = "diff_criteria" if "diff_criteria" in em_table else "diff"

nominal_summary = pd.DataFrame({
    "magnitud": ["Semilla (VWC %)", "Iteración final", "Chi2 final",
                 "Chi2 reducido", "diff final"],
    "valor": [
        nominal_row.get("seed_bin_edgeds", 100 * water_fraction),
        nominal_row["em_it"],
        nominal_row["Chi2"],
        nominal_row["Chi2red"],
        nominal_row[diff_branch],
    ],
})
display(nominal_summary)

fig, axis = plt.subplots()
axis.loglog(energy_mid, seed_integral, label="semilla")
axis.loglog(energy_mid, final_integral, label="EM nominal")
axis.set(xlabel="Energía [MeV]",
         ylabel=r"Flujo por bin [cm$^{-2}$ s$^{-1}$]",
         title=f"Espectro nominal · semilla {SEED_NUMBER}")
axis.legend()
plt.show()


## 8. ¿Por qué se detuvo?

El C++ itera mientras falle al menos una condición:

\[
\chi^2>N_{\mathrm{det}}\quad\text{o}\quad\mathrm{diff}>0.02.
\]

Por ello, la parada adoptada requiere simultáneamente:

\[
\boxed{\chi^2\leq N_{\mathrm{det}}\quad\text{y}\quad\mathrm{diff}\leq0.02}
\]

salvo que se alcance antes el máximo. El log oficial conserva ambos diagnósticos para cada iteración.


In [ ]:
history = em_iteration_history(em_log, SEED_NUMBER)
display(history)

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))
left.plot(history["iteración"], history["Chi2"], "o-")
left.axhline(int(config["NDET"]), color="tab:red", linestyle="--",
             label=f"NDET={config['NDET']}")
left.set(xlabel="Iteración", ylabel=r"$\chi^2$",
         title="Reproducción de los datos")
left.legend()

right.plot(history["iteración"], history["diff"], "o-")
right.axhline(0.02, color="tab:red", linestyle="--", label="0.02")
right.set(xlabel="Iteración", ylabel="diff",
          title="Estabilidad del espectro")
right.legend()
plt.show()

last = history.iloc[-1]
criteria_met = (
    last["Chi2"] <= int(config["NDET"]) and last["diff"] <= 0.02
)
print("Cumple ambos criterios:", criteria_met)
print("Iteración final:", int(last["iteración"]),
      "| máximo configurado:", config["MAX_STEPS"])


Más iteraciones no garantizan una solución más física. La parada funciona como regularización:

| Caso | Refolding | Forma esperada | Interpretación |
|---|---|---|---|
| Temprana | Deficiente | Cercana a la semilla | Subajuste |
| Adoptada | Compatible | Estable | Compromiso seleccionado |
| Tardía | Mejora pequeña | Puede volverse irregular | Riesgo de ajustar fluctuaciones |


## 9. Refolding: validar la solución

Volvemos a plegar el espectro nominal:

\[
\widehat n_i=\sum_jR_{ij}\phi_j\Delta E_j.
\]

Una solución aceptable debe reproducir las tasas dentro de sus incertidumbres, no solamente verse suave.


In [ ]:
refolded_rate = forward_fold(
    response_matrix, final_differential, energy_width
)
normalized_residual = (measured_rate - refolded_rate) / measured_sigma
refold_ratio = refolded_rate / measured_rate
refold_chi2 = np.sum(normalized_residual ** 2)

refold_table = event_table[["Canal", "Tasa [s⁻¹]",
                            "Incertidumbre [s⁻¹]"]].copy()
refold_table["Tasa refoldeada [s⁻¹]"] = refolded_rate
refold_table["Residuo normalizado"] = normalized_residual
refold_table["Refold / medida"] = refold_ratio
display(refold_table.style.format({
    "Tasa [s⁻¹]": "{:.5f}",
    "Incertidumbre [s⁻¹]": "{:.5f}",
    "Tasa refoldeada [s⁻¹]": "{:.5f}",
    "Residuo normalizado": "{:.2f}",
    "Refold / medida": "{:.3f}",
}))
print(f"χ² recalculado en Python: {refold_chi2:.5f}")
print(f"χ² guardado por ROOT    : {nominal_row['Chi2']:.5f}")


In [ ]:
x = np.arange(len(channels))
fig, (top, bottom) = plt.subplots(
    2, 1, figsize=(11, 7), sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
)
top.errorbar(x, measured_rate, yerr=measured_sigma, fmt="o",
             capsize=3, label="medida")
top.plot(x, refolded_rate, "s-", label="refolding EM")
top.set(ylabel="Tasa [s⁻¹]", title="Medición frente a refolding")
top.legend()
bottom.axhline(0, color="black", linestyle="--")
bottom.bar(x, normalized_residual, color="tab:purple", alpha=0.75)
bottom.axhline(1, color="tab:red", linestyle=":")
bottom.axhline(-1, color="tab:red", linestyle=":")
bottom.set(ylabel="Residuo / σ", xlabel="Canal activo",
           xticks=x, xticklabels=channels)
plt.show()


**Pregunta central:** ¿el espectro reconstruido reproduce todos los canales dentro de sus incertidumbres?


## 10. Incertidumbre mediante Monte Carlo

Para cada intento se fluctúan los conteos según el modelo implementado en C++, se ejecuta EM y se aplican criterios de aceptación:

```text
conteos → fluctuación → nuevo conjunto → EM
        → criterios → espectro aceptado o rechazado
```

| Nivel | Pregunta |
|---|---|
| Parada EM | ¿Cuándo dejamos de iterar una realización? |
| Aceptación MC | ¿La realización convergió lo suficiente para integrar la muestra? |

La aceptación exige `em_it < MC_MAX_STEPS`, `Chi2 < NDET` y `diff < DIFF_LIMIT`.

> **Advertencia:** la celda siguiente configura un MC reducido. No utiliza los 10 000 aceptados de producción y no se ejecuta con **Run All** hasta cambiar explícitamente `RUN_REDUCED_MC=True`.


In [ ]:
RUN_REDUCED_MC = False
TARGET_ACCEPTED = 500
MAX_MC_TRIALS = 200_000
MC_MAX_STEPS = 20
DIFF_LIMIT = 0.02

if TARGET_ACCEPTED > 500:
    raise ValueError("TARGET_ACCEPTED debe ser ≤ 500 en modo docente.")

mc_file = expected_mc_output(CONFIG_FILE, EVENT_ID)
mc_stats_file = expected_mc_stats(CONFIG_FILE, EVENT_ID)
mc_command = (
    "bash local_scripts/run_event_em_mc_local.sh "
    f"{CONFIG_FILE} {EVENT_ID} {TARGET_ACCEPTED} "
    f"{MAX_MC_TRIALS} {MC_MAX_STEPS} {DIFF_LIMIT}"
)
print("Comando MC :", mc_command)
print("Salida ROOT:", mc_file)
print("Resumen    :", mc_stats_file)

if RUN_REDUCED_MC:
    if REUSE_EXISTING_RESULTS and mc_file.is_file() and mc_stats_file.is_file():
        print("\nSe reutiliza el MC existente; revisa abajo su estadística real.")
    else:
        try:
            run_em_mc(
                CONFIG_FILE,
                EVENT_ID,
                target_accepted=TARGET_ACCEPTED,
                max_mc_trials=MAX_MC_TRIALS,
                mc_max_steps=MC_MAX_STEPS,
                diff_limit=DIFF_LIMIT,
            )
        except subprocess.CalledProcessError as error:
            print(f"\nEl MC terminó con código {error.returncode}.")
            print("Revisa outputs/log/local_em_mc/.")
            raise
else:
    print("\nMC no iniciado. Cambia RUN_REDUCED_MC=True para ejecutarlo.")


### Resultados aceptados y rechazados


In [ ]:
if mc_file.is_file():
    print("Árboles:", tree_names(mc_file))
    print("Entradas aceptadas:", tree_entry_count(mc_file))
    display(pd.DataFrame({"Branch": tree_branches(mc_file)}))
    mc_table = scalar_tree_table(mc_file, max_rows=None)
    mc_spectra = vector_tree_array(mc_file)
    mc_stats = mc_run_summary(CONFIG_FILE, EVENT_ID)
    display(pd.DataFrame([mc_stats]))
    print("Forma de los espectros MC:", mc_spectra.shape)
else:
    mc_table = pd.DataFrame()
    mc_spectra = np.empty((0, len(energy_mid)))
    mc_stats = None
    print("No existe un ROOT MC para este evento. Ejecuta la celda protegida.")


In [ ]:
if mc_stats is not None:
    fig, axis = plt.subplots(figsize=(7, 4))
    axis.bar(
        ["Aceptadas", "Rechazadas"],
        [mc_stats["aceptadas"], mc_stats["rechazadas"]],
        color=["tab:green", "tab:red"],
    )
    axis.set(ylabel="Intentos", title=(
        f"Fracción de aceptación = {mc_stats['fracción']:.1%} "
        f"({mc_stats['aceptadas']}/{mc_stats['intentos']})"
    ))
    plt.show()
else:
    print("Sin resumen MC.")


### Diagnósticos de las realizaciones aceptadas


In [ ]:
distribution_columns = [
    "Chi2", "diff", "em_it", "Intg_total",
    "Intg_th", "Intg_ep", "Intg_fs", "Intg_he",
]

if not mc_table.empty:
    available = [column for column in distribution_columns
                 if column in mc_table]
    fig, axes = plt.subplots(4, 2, figsize=(12, 14))
    for axis, column in zip(axes.flat, available):
        values = mc_table[column].dropna()
        axis.hist(values, bins=30, color="tab:blue", alpha=0.8)
        axis.axvline(values.median(), color="tab:red",
                     label="mediana")
        axis.set(title=column, xlabel=column,
                 ylabel="Realizaciones aceptadas")
        axis.legend()
    for axis in axes.flat[len(available):]:
        axis.set_visible(False)
    fig.tight_layout()
    plt.show()
    display(quantile_summary(mc_table, available))
else:
    print("Sin realizaciones MC para graficar.")


### Espectros aceptados y banda de incertidumbre

Se muestran como máximo 20 realizaciones. La banda usa \(q_{15.865}\), la mediana y \(q_{84.135}\) en cada bin; no se supone simetría.


In [ ]:
if len(mc_spectra):
    q_low, q_median, q_high = np.quantile(
        mc_spectra, [0.15865, 0.5, 0.84135], axis=0
    )
    sample_count = min(20, len(mc_spectra))
    sample_indices = np.linspace(
        0, len(mc_spectra) - 1, sample_count, dtype=int
    )

    fig, axis = plt.subplots()
    for index in sample_indices:
        axis.loglog(energy_mid, mc_spectra[index],
                    color="tab:blue", alpha=0.15)
    axis.loglog(energy_mid, q_median, color="black",
                linewidth=2, label="mediana")
    axis.set(xlabel="Energía [MeV]",
             ylabel=r"Flujo por bin [cm$^{-2}$ s$^{-1}$]",
             title=f"Muestra de {sample_count} espectros aceptados")
    axis.legend()
    plt.show()

    fig, axis = plt.subplots()
    axis.fill_between(energy_mid, q_low, q_high, color="tab:blue",
                      alpha=0.3, label="q15.865–q84.135")
    axis.loglog(energy_mid, q_median, color="tab:blue",
                label="mediana MC")
    axis.loglog(energy_mid, final_integral, color="black",
                linestyle="--", label="nominal")
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set(xlabel="Energía [MeV]",
             ylabel=r"Flujo por bin [cm$^{-2}$ s$^{-1}$]",
             title="Espectro con banda de incertidumbre")
    axis.legend()
    plt.show()
else:
    q_low = q_median = q_high = None
    print("La banda aparecerá después de generar o cargar el MC.")


> **Alcance de la banda:** describe la incertidumbre asociada al modelo de fluctuaciones usado por el Monte Carlo. No incluye automáticamente todas las incertidumbres sistemáticas de respuestas, calibraciones, hipótesis angular, semillas, tiempos vivos, selección de canales o eficiencia.


## 11. Flujos integrales y razón térmica

El C++ integra cada réplica antes de guardar `Intg_th`, `Intg_ep`, `Intg_fs`, `Intg_he` e `Intg_total`. Por ello, los cuantiles siguientes respetan las correlaciones entre bins.

\[
\mathcal R=\frac{\Phi_{\mathrm{th}}}
{\Phi_{\mathrm{ep}}+\Phi_{\mathrm{fast}}}
\]

también se calcula réplica a réplica, no como cociente de medianas.


In [ ]:
region_info = pd.DataFrame({
    "Región": ["Térmica", "Epitérmica", "Rápida",
               "Alta energía", "Total"],
    "Branch": ["Intg_th", "Intg_ep", "Intg_fs",
               "Intg_he", "Intg_total"],
    "Intervalo aproximado [MeV]": [
        "≤ 1.9×10⁻⁷", "2.2×10⁻⁷ – 9×10⁻³",
        "1.1×10⁻² – 8.9", "10.5 – 7.6×10³", "todos",
    ],
})

if not mc_table.empty:
    integral_quantiles = quantile_summary(
        mc_table, region_info["Branch"]
    ).reset_index(names="Branch")
    integral_table = region_info.merge(
        integral_quantiles, on="Branch", how="left"
    )[[
        "Región", "Intervalo aproximado [MeV]", "mediana",
        "incertidumbre_-", "incertidumbre_+",
    ]]
    display(integral_table.style.format({
        "mediana": "{:.6g}",
        "incertidumbre_-": "{:.3g}",
        "incertidumbre_+": "{:.3g}",
    }))

    thermal_ratio = (
        mc_table["Intg_th"]
        / (mc_table["Intg_ep"] + mc_table["Intg_fs"])
    )
    ratio_quantiles = thermal_ratio.quantile(
        [0.15865, 0.5, 0.84135]
    )
    ratio_summary = {
        "mediana": ratio_quantiles.loc[0.5],
        "incertidumbre_-": (
            ratio_quantiles.loc[0.5] - ratio_quantiles.loc[0.15865]
        ),
        "incertidumbre_+": (
            ratio_quantiles.loc[0.84135] - ratio_quantiles.loc[0.5]
        ),
    }
    display(pd.DataFrame([ratio_summary], index=["Razón térmica"]))

    fig, axis = plt.subplots(figsize=(8, 4))
    axis.hist(thermal_ratio, bins=30, color="tab:orange", alpha=0.8)
    axis.axvline(ratio_summary["mediana"], color="black",
                 label="mediana")
    axis.set(xlabel=r"$\Phi_{th}/(\Phi_{ep}+\Phi_{fast})$",
             ylabel="Realizaciones", title="Razón térmica")
    axis.legend()
    plt.show()
else:
    integral_table = pd.DataFrame()
    thermal_ratio = pd.Series(dtype=float)
    ratio_summary = None
    print("Los flujos y la razón aparecerán después del MC.")


## 12. Comparación opcional con alta estadística

El docente puede colocar un ROOT de referencia en la ruta siguiente. Si no está disponible, el notebook continúa sin error. El archivo debe contener `em_loop_tree/deconv_vec` para el mismo evento y configuración.


In [ ]:
REFERENCE_MC_FILE = (
    REPO_DIR / "data/reference"
    / f"{config['CAMPAIGN']}_event_{EVENT_ID}_high_statistics.root"
)

if REFERENCE_MC_FILE.is_file() and len(mc_spectra):
    reference_spectra = vector_tree_array(REFERENCE_MC_FILE)
    ref_low, ref_median, ref_high = np.quantile(
        reference_spectra, [0.15865, 0.5, 0.84135], axis=0
    )
    fig, axis = plt.subplots()
    axis.fill_between(energy_mid, q_low, q_high,
                      color="tab:blue", alpha=0.25,
                      label=f"docente (n={len(mc_spectra)})")
    axis.loglog(energy_mid, q_median, color="tab:blue")
    axis.fill_between(energy_mid, ref_low, ref_high,
                      color="tab:orange", alpha=0.22,
                      label=f"referencia (n={len(reference_spectra)})")
    axis.loglog(energy_mid, ref_median, color="tab:orange")
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set(xlabel="Energía [MeV]",
             ylabel=r"Flujo por bin [cm$^{-2}$ s$^{-1}$]",
             title="MC docente frente a referencia")
    axis.legend()
    plt.show()
else:
    print("Referencia de alta estadística no disponible en:")
    print(REFERENCE_MC_FILE)
    print("La comparación queda preparada para cuando se agregue el ROOT.")


## 13. Resumen automático del evento


In [ ]:
if not integral_table.empty:
    integral_values = integral_table.set_index("Región")["mediana"]
else:
    integral_values = pd.Series(dtype=float)

summary_rows = {
    "Campaña": config["CAMPAIGN"],
    "Evento": EVENT_ID,
    "Número de detectores": config["NDET"],
    "Tiempo de integración [min]": config["TIME_GRID"],
    "Semilla": SEED_NUMBER,
    "VWC semilla [%]": 100 * water_fraction,
    "Iteración de parada": int(nominal_row["em_it"]),
    "Chi2 final": float(nominal_row["Chi2"]),
    "diff final": float(nominal_row[diff_branch]),
    "Intentos MC": mc_stats["intentos"] if mc_stats else "no ejecutado",
    "Realizaciones aceptadas": (
        mc_stats["aceptadas"] if mc_stats else "no ejecutado"
    ),
    "Fracción de aceptación": (
        mc_stats["fracción"] if mc_stats else np.nan
    ),
    "Flujo térmico": integral_values.get("Térmica", np.nan),
    "Flujo epitérmico": integral_values.get("Epitérmica", np.nan),
    "Flujo rápido": integral_values.get("Rápida", np.nan),
    "Flujo de alta energía": integral_values.get("Alta energía", np.nan),
    "Flujo total": integral_values.get("Total", np.nan),
    "Razón térmica": (
        ratio_summary["mediana"] if ratio_summary else np.nan
    ),
}
display(pd.DataFrame(
    {"Magnitud": summary_rows.keys(), "Valor": summary_rows.values()}
))


## Cierre

**Pregunta final:** ¿la solución reproduce adecuadamente las tasas medidas y qué región del espectro presenta mayor incertidumbre?

La idea principal es que el espectro no se mide directamente; se infiere y se valida mediante:

\[
\boxed{\text{datos}+R+\text{algoritmo}+\text{parada}
+\text{propagación de incertidumbre}}
\]

Para comparar una segunda condición física, conviene **cargar un segundo evento precalculado** y contrastar sus flujos integrales o su razón térmica, sin iniciar otra ejecución extensa durante la clase.
